In [1]:
# BUSINESS SCIENCE -----
# LAB 91: CUSTOMER LIFETIME VALUE -----
# Part 1: DESCRIPTIVE CLV MODELS (BEGINNER) -----
# *** -----

# BUSINESS GOALS:
# 1. HOW MUCH CAN WE SPEND TO ACQUIRE A CUSTOMER? LIFETIME VALUE - COST TO ACQUIRE CUSTOMER (LTV - CAC)
# 2. WHICH CUSTOMERS ARE MOST VALUABLE? SEGMENTATION
# 3. HOW CAN WE INCREASE CUSTOMER LIFETIME VALUE? MARKETING STRATEGY

# 3 PARTS:
# 1. Descriptive CLV Models (covered in this script)
# 2. Probabilistic CLV Models (covered in Part 2)
# 3. Predictive CLV Models (covered in Part 3)
# * Stick around for Part 3, that's where we'll save the MOST MONEY!

In [2]:
# LIBRARIES -----

import pandas as pd
import pytimetk as tk



c:\Users\mmopa\miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-15 15:01:10,192	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:

# DATA -----


transactions_df = pd.read_csv('data_transactions_processed.csv')

transactions_df.glimpse()

df = transactions_df.copy()

df['timestamp'] = pd.to_datetime(df['timestamp'])

<class 'pandas.core.frame.DataFrame'>: 300000 rows of 23 columns
household_key:        int64             [103, 436, 349, 271, 107, 72, 18 ...
basket_id:            int64             [1537972, 1401354, 1293144, 1656 ...
day:                  int64             [78, 356, 121, 288, 649, 213, 64 ...
product_id:           int64             [549, 47, 468, 297, 514, 418, 36 ...
quantity:             int64             [6, 8, 1, 5, 4, 6, 8, 9, 9, 4, 1 ...
sales_value:          float64           [15.08, 17.13, 12.03, 7.73, 3.91 ...
store_id:             int64             [49, 36, 24, 6, 49, 29, 7, 20, 4 ...
retail_disc:          float64           [4.24, 0.84, 3.48, 2.12, 4.85, 2 ...
trans_time:           int64             [319, 411, 812, 113, 1051, 356,  ...
week_no:              int64             [12, 51, 18, 42, 93, 31, 92, 49, ...
coupon_disc:          float64           [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0 ...
coupon_match_disc:    float64           [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0 ...
commodity_d

In [5]:
df.head()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,...,department,brand,age_desc,marital_status_code,income_desc,homeowner_desc,hh_comp_desc,household_size_desc,kid_category_desc,timestamp
0,103,1537972,78,549,6,15.08,49,4.24,319,12,...,PASTRY,Private,19-24,B,Under 15K,Unknown,Unknown,1,None/Unknown,2023-03-20
1,436,1401354,356,47,8,17.13,36,0.84,411,51,...,NUTRITION,National,25-34,A,125-149K,Homeowner,Unknown,1,None/Unknown,2023-12-23
2,349,1293144,121,468,1,12.03,24,3.48,812,18,...,MEAT,Private,45-54,A,150-174K,Homeowner,Unknown,5,None/Unknown,2023-05-02
3,271,1656002,288,297,5,7.73,6,2.12,113,42,...,NUTRITION,National,19-24,B,175-199K,Unknown,Unknown,2,None/Unknown,2023-10-16
4,107,1446178,649,514,4,3.91,49,4.85,1051,93,...,PASTRY,National,35-44,A,25-34K,Renter,Unknown,3,None/Unknown,2024-10-11


In [7]:

# EXPLORATORY DATA ANALYSIS -----

df['sales_value'].sum()

df[['timestamp', 'sales_value']] \
    .summarize_by_time(
        date_column = 'timestamp',
        value_column = 'sales_value',
        agg_func = 'sum',
        freq = 'M'
    ) \
    .plot_timeseries('timestamp', 'sales_value')

In [13]:

# 1.0 AGGREGATION MODELS -----

# Aggregation models are used to calculate the average customer
# lifetime value for a group of customers or a cohort.

customer_sales_1_df = df \
    .groupby(['household_key', 'basket_id']) \
    .agg(
        total_sales_basket=('sales_value', 'sum'),
        timestamp=('timestamp', 'max')
    ) \
    .reset_index() \
    .groupby('household_key') \
    .agg(
        # Time difference in days
        time_days=('timestamp', lambda x: (x.max() - x.min()).days),
        # Count of unique 'basket_id'
        frequency=('basket_id', 'nunique'),
        # Sum of 'sales_value'
        total_sales=('total_sales_basket', 'sum'),
        avg_sales=('total_sales_basket', 'mean')
    ) \
    .reset_index()

customer_sales_1_df

,household_key,time_days,frequency,total_sales,avg_sales
0,1,707,591,6111.46,10.340880
1,2,709,618,5973.67,9.666133
2,3,709,580,5657.83,9.754879
3,4,710,603,6147.37,10.194643
4,5,709,589,5631.52,9.561154
...,...,...,...,...,...
495,496,708,613,6285.66,10.253931
496,497,710,605,6217.26,10.276463
497,498,704,552,5327.19,9.650707
498,499,710,577,5669.37,9.825598


In [14]:
summary_1 = {
    'average_sales': customer_sales_1_df['avg_sales'].mean(),
    'average_purchase_freq': customer_sales_1_df['frequency'].mean(),
    'churn_rate': 1 - (customer_sales_1_df['frequency'] > 5).sum() / len(customer_sales_1_df['frequency']),
    'max_days': customer_sales_1_df['time_days'].max()
}

summary_1_df = pd.DataFrame([summary_1])

summary_1_df

,average_sales,average_purchase_freq,churn_rate,max_days
0,10.00575,599.81,0.0,710


In [15]:

# Define the constants
profit_margin     = 0.15  # 15% Profit on Products
customer_lifetime = 5     # 5 years
eps_churn_rate    = 0.001


In [16]:

# Churn CLV Calculation
summary_1_df['clv_churn_method'] = (summary_1_df['average_sales'] 
    * summary_1_df['average_purchase_freq'] / (summary_1_df['churn_rate'] 
    + eps_churn_rate)) * profit_margin

# Lifetime CLV Calculation
summary_1_df['clv_lifetime_method'] = (summary_1_df['average_sales'] 
    * summary_1_df['average_purchase_freq'] 
    * (summary_1_df['max_days'] / 365) * customer_lifetime) * profit_margin

summary_1_df

,average_sales,average_purchase_freq,churn_rate,max_days,clv_churn_method,clv_lifetime_method
0,10.00575,599.81,0.0,710,900232.373423,8755.684728


In [17]:
# 2.0 COHORT MODELS -----

# Cohort models are used to calculate the average customer
# lifetime value for a group of customers or a cohort. Often times,
# the cohort is defined by the customer's first purchase date.

# Constants
profit_margin     = 0.15  # 15% Profit on Products
customer_lifetime = 5     # 5 years
eps_churn_rate    = 0.001

In [18]:

# Calculate start_month for each household
df['start_month'] = df.groupby('household_key')['timestamp'] \
    .transform(lambda x: x.min().strftime('%Y-%m'))

df.glimpse()


<class 'pandas.core.frame.DataFrame'>: 300000 rows of 24 columns
household_key:        int64             [103, 436, 349, 271, 107, 72, 18 ...
basket_id:            int64             [1537972, 1401354, 1293144, 1656 ...
day:                  int64             [78, 356, 121, 288, 649, 213, 64 ...
product_id:           int64             [549, 47, 468, 297, 514, 418, 36 ...
quantity:             int64             [6, 8, 1, 5, 4, 6, 8, 9, 9, 4, 1 ...
sales_value:          float64           [15.08, 17.13, 12.03, 7.73, 3.91 ...
store_id:             int64             [49, 36, 24, 6, 49, 29, 7, 20, 4 ...
retail_disc:          float64           [4.24, 0.84, 3.48, 2.12, 4.85, 2 ...
trans_time:           int64             [319, 411, 812, 113, 1051, 356,  ...
week_no:              int64             [12, 51, 18, 42, 93, 31, 92, 49, ...
coupon_disc:          float64           [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0 ...
coupon_match_disc:    float64           [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0 ...
commodity_d

In [19]:
df.head()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,...,brand,age_desc,marital_status_code,income_desc,homeowner_desc,hh_comp_desc,household_size_desc,kid_category_desc,timestamp,start_month
0,103,1537972,78,549,6,15.08,49,4.24,319,12,...,Private,19-24,B,Under 15K,Unknown,Unknown,1,None/Unknown,2023-03-20,2023-01
1,436,1401354,356,47,8,17.13,36,0.84,411,51,...,National,25-34,A,125-149K,Homeowner,Unknown,1,None/Unknown,2023-12-23,2023-01
2,349,1293144,121,468,1,12.03,24,3.48,812,18,...,Private,45-54,A,150-174K,Homeowner,Unknown,5,None/Unknown,2023-05-02,2023-01
3,271,1656002,288,297,5,7.73,6,2.12,113,42,...,National,19-24,B,175-199K,Unknown,Unknown,2,None/Unknown,2023-10-16,2023-01
4,107,1446178,649,514,4,3.91,49,4.85,1051,93,...,National,35-44,A,25-34K,Renter,Unknown,3,None/Unknown,2024-10-11,2023-01


In [20]:
# Aggregate data by start_month and household_key
cohort_data = df \
    .groupby(['start_month', 'household_key', 'basket_id']) \
    .agg(
        total_sales_basket=('sales_value', 'sum'),
        timestamp=('timestamp', 'max')
    ) \
    .reset_index() \
    .groupby(['start_month', 'household_key']) \
    .agg(
        time_days=('timestamp', lambda x: (x.max() - x.min()).days),
        frequency=('basket_id', 'nunique'),
        total_sales=('total_sales_basket', 'sum'),
        avg_sales=('total_sales_basket', 'mean')
    ) \
    .reset_index()


In [21]:
# Calculate CLV metrics by start_month
summary_2_df = cohort_data \
    .groupby('start_month') \
    .agg(
        cohort_size=('household_key', 'nunique'),
        average_sales=('avg_sales', 'mean'),
        average_purchase_freq=('frequency', 'mean'),
        churn_rate=('frequency', lambda x: 1 - (x > 5).sum() / len(x)),
        max_days=('time_days', 'max')
    ) \
    .reset_index()


In [22]:

# Add Churn CLV calculation
summary_2_df['clv_churn_method'] = (summary_2_df['average_sales'] 
    * summary_2_df['average_purchase_freq'] / (summary_2_df['churn_rate'] 
    + eps_churn_rate)) * profit_margin


In [23]:
# Add Lifetime CLV calculation
summary_2_df['clv_lifetime_method'] = (summary_2_df['average_sales'] 
    * summary_2_df['average_purchase_freq'] 
    * (summary_2_df['max_days'] / 365) * customer_lifetime) * profit_margin

summary_2_df


,start_month,cohort_size,average_sales,average_purchase_freq,churn_rate,max_days,clv_churn_method,clv_lifetime_method
0,2023-01,500,10.00575,599.81,0.0,710,900232.373423,8755.684728


In [24]:
# CONCLUSIONS -----
# 1. I don't trust these Customer Lifetime Value (CLV) calculations. 
#    They are super optimistic for the churn calculation. 
#    Lifetime is a bit more realistic, but still has not earned my trust.
# 2. The bottom line is that as we get more granular, we can gain 
#    higher accuracy in our CLV calculations.
# 3. The next step is to build predictive models to forecast future 
#    customer lifetime value.